# 03 — Biomni-Eval1 Dataset Evaluation

This notebook mirrors `pipelines/run_eval1.py`. It runs the [biomni/Eval1](https://huggingface.co/datasets/biomni/Eval1) benchmark — **433 expert-curated biological reasoning instances** across 10 task types — against any framework and model.

### Dataset at a glance

| Task type | What it tests |
|---|---|
| `gwas_causal_gene_opentargets` | Identify causal gene in a GWAS locus (OpenTargets) |
| `gwas_causal_gene_abc` | Identify causal gene (ABC model) |
| `gwas_causal_gene_ot_genetics` | Identify causal gene (OT Genetics) |
| `gwas_variant_prioritization` | Prioritise disease-associated variants |
| `screen_gene_retrieval` | Retrieve genes from functional screens |
| `lab_bench_seqqa` | Sequencing-related multiple-choice questions |
| `lab_bench_cloningqa` | Cloning-related multiple-choice questions |
| `rare_disease_diagnosis` | Diagnose rare disease from HPO phenotypes + gene |
| `crispr_delivery` | Select optimal CRISPR delivery method |
| `patient_gene_detection` | Detect causal gene from patient data |

Scoring is **binary exact match** (0.0 or 1.0) via `BiomniEval1().evaluate()`.

### Prerequisites
```bash
uv sync --extra biomni --extra dev
ollama pull llama3
```

In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Repo root: {repo_root}")

## 2. Explore the dataset

Load the full dataset directly from HuggingFace and inspect its structure.

In [ ]:
from datasets import load_dataset

ds = load_dataset("biomni/Eval1", split="test")
df_full = ds.to_pandas()

print(f"Total instances: {len(df_full)}")
print(f"Columns: {list(df_full.columns)}")
print()
print("Instances per task type:")
print(df_full["task_name"].value_counts().to_string())

In [ ]:
# Preview a sample instance from each task type
df_full.groupby("task_name").first()[["prompt", "answer"]].head(10)

In [ ]:
# Look at a specific instance in full
TASK_TO_PREVIEW = "gwas_causal_gene_opentargets"

row = df_full[df_full["task_name"] == TASK_TO_PREVIEW].iloc[0]
print(f"task_name       : {row['task_name']}")
print(f"task_instance_id: {row['task_instance_id']}")
print(f"answer          : {row['answer']}")
print(f"\n--- Prompt ---\n{row['prompt']}")

## 3. Configure the evaluation run

In [ ]:
from bio_agents.evaluation.eval1_runner import Eval1Config

# Option A — load from YAML (recommended)
CONFIG_PATH = repo_root / "configs" / "biomni_eval1_smoke.yaml"
cfg = Eval1Config.from_yaml(CONFIG_PATH)

# Option B — define inline (uncomment to use)
# cfg = Eval1Config(
#     task_names=[
#         "gwas_causal_gene_opentargets",
#         "lab_bench_seqqa",
#         "rare_disease_diagnosis",
#     ],
#     instances_per_task=1,
#     frameworks=["biomni"],
#     models=["llama3"],
#     output_dir="results/biomni_eval1_notebook",
#     runner_kwargs={"use_tool_retriever": True, "timeout_seconds": 300},
# )

print(f"Task types     : {cfg.resolved_task_names()}")
print(f"Instances/task : {cfg.instances_per_task}")
print(f"Frameworks     : {cfg.frameworks}")
print(f"Models         : {cfg.models}")
print(f"Output dir     : {cfg.output_dir}")

## 4. Preview the matrix (dry-run)

The `Eval1Runner` loads real `task_instance_id` values from the dataset, so the dry-run shows exactly which instances will be evaluated.

In [ ]:
from bio_agents.evaluation.eval1_runner import Eval1Runner

runner = Eval1Runner(cfg)
plan = runner.dry_run()

df_plan = pd.DataFrame(plan)
print(f"Total runs planned: {len(df_plan)}")
df_plan

## 5. Run the evaluation

Each instance is sent to the agent as a prompt. The response is scored by `BiomniEval1().evaluate()` using exact match against the ground truth answer.

In [ ]:
import time

print(f"Running {len(plan)} eval instance(s)... (may take several minutes)")
t0 = time.perf_counter()
results = runner.run()
elapsed = time.perf_counter() - t0

passed = sum(r.eval_result.passed for r in results)
avg_score = sum(r.eval_result.score for r in results) / len(results)
print(f"Done in {elapsed:.1f}s")
print(f"Passed: {passed}/{len(results)}  |  Avg score: {avg_score:.3f}")

## 6. Analyse results

In [ ]:
df = pd.DataFrame([r.to_dict() for r in results])
df[
    [
        "eval1_task_name",
        "task_instance_id",
        "framework",
        "model",
        "score",
        "passed",
        "duration_s",
    ]
]

In [ ]:
# Score per task type
df.groupby("eval1_task_name")[["score", "passed"]].agg(
    avg_score=("score", "mean"),
    n_passed=("passed", "sum"),
    n_total=("passed", "count"),
).assign(pass_rate=lambda x: x["n_passed"] / x["n_total"]).sort_values(
    "avg_score", ascending=False
)

In [ ]:
# Inspect a single result in detail
r = results[0]
print(f"Task type      : {r.eval1_task_name}")
print(f"Instance ID    : {r.task_instance_id}")
print(f"Score          : {r.eval_result.score}")
print(f"Passed         : {r.eval_result.passed}")
print(f"Eval metrics   : {r.eval_result.metrics}")
print("\n--- Agent output (first 1000 chars) ---")
print((r.agent_result.output or "")[:1000])

## 7. Save and reload results

In [ ]:
out_file = runner.save(results)
print(f"Saved to: {out_file}")

In [ ]:
# Reload from JSONL — useful for comparing runs across sessions
df_saved = pd.read_json(out_file, lines=True)
print(f"Rows in file: {len(df_saved)}")

# Overall score per model (useful when running multiple models)
df_saved.groupby(["framework", "model"])["score"].agg(["mean", "count"]).rename(
    columns={"mean": "avg_score", "count": "n_runs"}
)

## 8. Scale up

To run more instances or all task types, edit `configs/biomni_eval1_smoke.yaml`:

```yaml
task_names:
  - all            # run all 10 task types

instances_per_task: 5   # run 5 instances per task = 50 total
```

Or switch to the full config:
```python
cfg = Eval1Config.from_yaml(repo_root / "configs" / "biomni_eval1_full.yaml")
```

For better scores, switch to a stronger model:
```python
# In the config cell above:
cfg.models = ["llama-3.3-70b"]   # Groq free tier — GROQ_API_KEY required
```